# Iceberg and ship classification

Train the modern two-band CNN on the fixed comparison split, evaluate the saved checkpoint,
and produce a complete competition submission. The attached code dataset contains a wheel,
experiment configurations and their source fingerprints. Internet and private tracking services
are not required. Hosted dependencies are recorded separately from the local lock.

The historical single-model public benchmark is **0.19348**. The notebook reports local holdout
loss; an actual competition submission provides the leaderboard score.

In [ ]:
import hashlib
import importlib.metadata as metadata
import json
from pathlib import Path
import subprocess
import sys

from packaging.requirements import Requirement
from packaging.specifiers import SpecifierSet

wheels = list(Path("/kaggle/input").rglob("iceberg_classifier-*.whl"))
assert len(wheels) == 1, f"Attach one code wheel after its dataset is ready; found {len(wheels)}"
wheel = wheels[0]
code_dir = wheel.parent
code_source = json.loads((code_dir / "source.json").read_text())
assert hashlib.sha256(wheel.read_bytes()).hexdigest() == code_source["sha256"]
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", str(wheel)], check=True)
package = metadata.metadata("iceberg-classifier")
assert metadata.version("iceberg-classifier")
assert SpecifierSet(package["Requires-Python"]).contains(".".join(map(str, sys.version_info[:3])))
for declaration in metadata.requires("iceberg-classifier") or []:
    requirement = Requirement(declaration)
    if requirement.marker is None or requirement.marker.evaluate():
        assert requirement.specifier.contains(metadata.version(requirement.name)), declaration

from iceberg.data import sha256
from iceberg.runtime import source_identity
assert source_identity()["python_source_sha256"] == code_source["python_source_sha256"]
print({"code_commit": code_source["git_commit"], "wheel_sha256": code_source["sha256"],
       "python": sys.version.split()[0]})

In [ ]:
import shutil
from pathlib import PurePosixPath

competition = "statoil-iceberg-classifier-challenge"
roots = [Path("/kaggle/input/competitions") / competition, Path("/kaggle/input") / competition]
data_root = next((p for p in roots if p.is_dir()), None)
if data_root is None:
    candidates = list(Path("/kaggle/input").rglob("train.json.7z"))
    assert len(candidates) == 1, "Attach the official competition data"
    data_root = candidates[0].parent
extracted = Path("/kaggle/working/input-data")
extracted.mkdir(exist_ok=True)
for filename in ["train.json", "test.json"]:
    payloads = list(data_root.rglob(filename))
    if not payloads:
        archives = list(data_root.rglob(filename + ".7z"))
        assert len(archives) == 1, f"Expected one official {filename}.7z"
        try:
            import py7zr
        except ImportError:
            seven_zip = shutil.which("7z") or shutil.which("7zz") or shutil.which("7za") or shutil.which("7zr")
            assert seven_zip, "The offline runtime requires py7zr or the 7z executable"
            subprocess.run([seven_zip, "x", "-y", str(archives[0]), f"-o{extracted}"],
                           check=True, stdout=subprocess.DEVNULL)
        else:
            with py7zr.SevenZipFile(archives[0]) as archive:
                assert all(not PurePosixPath(n).is_absolute() and ".." not in PurePosixPath(n).parts
                           for n in archive.getnames())
                archive.extractall(extracted)
        payloads = list(extracted.rglob(filename))
    assert len(payloads) == 1, f"Expected one extracted {filename}"
    destination = extracted / filename
    if payloads[0].resolve() != destination.resolve():
        shutil.copy2(payloads[0], destination)
data_dir = extracted
print({name: sha256(data_dir / name) for name in ["train.json", "test.json"]})

In [ ]:
import zipfile
from iceberg.runtime import train, predict, verify_run, write_json

profile = "metadata"  # Choose "baseline" for the image-only comparison.
config_path = code_dir / f"config-{profile}.json"
assert sha256(config_path) == code_source["config_sha256"][config_path.name]
config = json.loads(config_path.read_text())
run_dir = Path("/kaggle/working/iceberg-run")
manifest = train(data_dir / "train.json", run_dir, config)
manifest["code_bundle"] = code_source
write_json(run_dir / "manifest.json", manifest)
verify_run(run_dir, data_dir / "train.json")
receipt = predict(run_dir / "model.pt", data_dir / "test.json",
                  Path("/kaggle/working/submission.csv"))
write_json(run_dir / "code-source.json", code_source)
with zipfile.ZipFile("/kaggle/working/run-bundle.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in run_dir.iterdir():
        if path.is_file():
            bundle.write(path, path.name)
    for name in ["submission.csv", "submission.receipt.json"]:
        bundle.write(Path("/kaggle/working") / name, name)
print(json.dumps({"run_id": manifest["logical_run_id"],
                  "validation": json.loads((run_dir / "metrics.json").read_text()),
                  "prediction": receipt}, indent=2))